# 🔬 파라미터 실험 — temperature · max_tokens · top_p

---

## 🎯 오늘의 상황

> **법률 AI 스타트업**에 개발자로 입사했습니다.  
> 팀장이 말합니다: "GPT로 법률 Q&A 챗봇을 만드는데, 파라미터를 어떻게 설정해야 할지 모르겠어요.  
> 같은 질문인데 매번 다른 답이 나와요. 이게 버그인가요?"
>  
> 오늘은 파라미터를 **직접 실험**하며 이 질문에 답해봅니다.

---

## 📋 실험 순서

```
실험 1-A. temperature 4단계 비교
실험 1-B. temperature=0.0 반복 3회 (동일한가?)
실험 1-C. max_tokens에 따른 응답 변화
실험 1-D. top_p 3단계 비교
실험 1-E. 환각(Hallucination) 관찰
```

---

## 💡 핵심 개념 — 파라미터 역할 구분

| 파라미터 | 역할 | 비유 |
|---------|------|------|
| `temperature` | 응답 다양성 조절 | 식당 선택 폭 (단골집 vs 눈감고 들어가기) |
| `max_tokens` | 응답 길이 상한선 | "이 이상은 말하지 마" |
| `top_p` | 단어 후보 범위 제한 | 주사위 눈 수 줄이기 |

> ⚠️ **실험의 원칙: 한 번에 하나씩만 바꾼다!**  
> 여러 개를 동시에 바꾸면 무엇 때문에 달라졌는지 알 수 없습니다.

---
## 🔧 사전 준비

In [1]:
# 라이브러리 불러오기 + 클라이언트 생성
from google.colab import userdata
import openai

client = openai.OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# 비용 추적기 (Day 1에서 만든 것)
class TokenTracker:
    def __init__(self):
        self.total_input  = 0
        self.total_output = 0
        self.call_count   = 0

    def add(self, response):
        self.total_input  += response.usage.prompt_tokens
        self.total_output += response.usage.completion_tokens
        self.call_count   += 1

    def report(self):
        cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"총 {self.call_count}회 호출 | 입력 {self.total_input} + 출력 {self.total_output} 토큰")
        print(f"누적 비용: ${cost:.4f} (약 {cost * 1400:.0f}원)")

tracker = TokenTracker()
print("✅ 준비 완료!")

✅ 준비 완료!


---
## 실험 전 예측 ✍️

실험 **전에** 결과를 예측해보세요. 예측과 실제 결과를 비교하면 기억에 더 오래 남습니다.

| 질문 | 내 예측 |
|------|--------|
| temperature=0.0 같은 질문 3번 → 결과가 같을까, 다를까? | |
| temperature=1.5에서 어떤 이상한 점이 생길까? | |
| 법률 Q&A에 가장 적합한 temperature는? | |
| max_tokens=30이면 어떤 일이 생길까? | |

---
## 실험 1-A. temperature 4단계 비교

같은 질문을 temperature만 바꿔가며 4번 호출합니다.  
응답이 어떻게 달라지는지 관찰하세요.

| temperature | 특성 | 어울리는 상황 |
|------------|------|---------------|
| `0.0` | 항상 같은 답 | 데이터 추출, 분류 |
| `0.5` | 약간의 변동 | **법률·의료 Q&A** |
| `1.0` | 창의적·다양 | 글쓰기, 아이디어 |
| `1.5` | 무작위 (환각 위험) | 실험적 창작 |

In [2]:
# 모든 실험에서 질문은 이 변수를 사용합니다 (고정)
question = "법원 판결문 작성 시 가장 중요한 요소는?"

# temperature를 0.0 → 0.5 → 1.0 → 1.5 순서로 바꿔가며 호출
for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n{'='*50}")
    print(f"[temperature = {temp}]")
    print(f"{'='*50}")

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=temp,   # ← 이 값만 바뀝니다
        max_tokens=100,     # 고정
        top_p=0.9,          # 고정
    )

    print(resp.choices[0].message.content)
    print(f"\n  → 종료 이유: {resp.choices[0].finish_reason}")
    print(f"  → 출력 토큰: {resp.usage.completion_tokens}")

    tracker.add(resp)   # 비용 추적


[temperature = 0.0]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확한 진술**: 사건의 경과와 관련된 사실을 정확하고 구체적으로 서술해야 합니다. 이는 판결의 기초가 되는 중요한 부분입니다.

2. **법리의 적용**: 사건에 적용되는 법률 조항과 판례를 명확히 하고, 이를 사건의 사실관계에 어떻게 적용했는지를 설명

  → 종료 이유: length
  → 출력 토큰: 100

[temperature = 0.5]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확한 기재**: 사건의 배경과 관련된 사실관계를 정확하고 명확하게 서술해야 합니다. 이를 통해 판결의 근거를 이해할 수 있도록 해야 합니다.

2. **법률적 근거**: 판결의 근거가 되는 법률 조항이나 판례를 명시해야 합니다. 이를 통해 판결이

  → 종료 이유: length
  → 출력 토큰: 100

[temperature = 1.0]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확한 기술**: 사건의 배경, 당사자, 주장 및 증거 등을 명확하게 정리하여 판결의 기초가 되는 사실관계를 정확히 서술해야 합니다.

2. **법률적 근거**: 판결의 근거가 되는 법률 조항이나 판례를 명확히 인용하여 법리적

  → 종료 이유: length
  → 출력 토큰: 100

[temperature = 1.5]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **명확성과 일관성**: 판결문은 법리와 사실을 명확하고 일관되게 전달해야 합니다. 법원은 판결의 이유를 분명하게 설명해야 하며, 법리가 어떻게 적용되었는지 잘 드러나야 합니다.

2. **법적 근거**: 판결문은 해당 사건에 적용된 법률과 판례

  → 종료 이유: length
  → 출력 토큰: 100


**📝 실험 1-A 기록표 — 직접 채워보세요**

| temperature | 응답이 일관적인가? | 어휘가 다양한가? | 법률 Q&A 적합도 | 특이사항 |
|------------|-----------------|----------------|----------------|----------|
| `0.0` | | | | |
| `0.5` | | | | |
| `1.0` | | | | |
| `1.5` | | | | |

---
## 실험 1-B. temperature=0.0 반복 3회

temperature=0.0에서는 정말 항상 같은 답이 나올까요?  
같은 설정으로 3번 호출해서 확인해봅니다.

In [3]:
print("─── temperature=0.0 반복 실험 (3회) ───")

responses_0 = []   # 3회 응답을 저장할 리스트

for i in range(3):   # 3번 반복
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0.0,   # 고정
        max_tokens=100,
    )
    answer = resp.choices[0].message.content
    responses_0.append(answer)   # 리스트에 저장
    print(f"\n[{i+1}회차]")
    print(answer[:120])   # 앞 120자만 출력
    tracker.add(resp)

# 3회 응답이 모두 동일한지 확인
if responses_0[0] == responses_0[1] == responses_0[2]:
    print("\n✅ 3회 모두 동일한 응답! (temperature=0.0은 결정론적)")
else:
    print("\n⚠️ 약간의 차이 발생 (드물지만 가능)")

─── temperature=0.0 반복 실험 (3회) ───

[1회차]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확한 진술**: 사건의 경과와 관련된 사실을 정확하고 구체적으로 서술해야 합니다. 이는 판결의 기초가 되는 중요한 요소입니다.

2.

[2회차]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확한 진술**: 사건의 경과와 관련된 사실을 정확하고 구체적으로 서술해야 합니다. 이는 판결의 기초가 되는 중요한 부분입니다.

2.

[3회차]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확한 진술**: 사건의 경과와 관련된 사실을 정확하고 구체적으로 서술해야 합니다. 이는 판결의 기초가 되는 중요한 요소입니다.

2.

⚠️ 약간의 차이 발생 (드물지만 가능)


---
## 실험 1-C. max_tokens에 따른 응답 변화

`max_tokens`는 **"이 이상은 말하지 마"** 라는 상한선입니다.  
너무 작으면 문장이 중간에 잘립니다. `finish_reason`이 `length`로 바뀌는 걸 확인하세요.

```
finish_reason: stop   → 모델이 자연스럽게 마무리 ✅
finish_reason: length → max_tokens에 걸려 강제 중단 ⚠️
```

In [4]:
# max_tokens를 30 → 100 → 300 순서로 바꿔가며 호출
q_long = "법원 판결문 작성 시 중요한 요소를 5가지 설명해 주세요."

for max_tok in [30, 100, 300]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": q_long}],
        temperature=0.3,     # 고정
        max_tokens=max_tok,  # ← 이 값만 바뀝니다
    )

    finish = resp.choices[0].finish_reason
    used   = resp.usage.completion_tokens

    print(f"\n{'='*50}")
    print(f"[max_tokens={max_tok}]  실제 사용: {used}토큰  종료: {finish}")
    print(f"{'='*50}")
    print(resp.choices[0].message.content)

    if finish == "length":   # 잘린 경우 경고
        print("\n⚠️ 문장이 중간에 잘렸습니다! max_tokens를 늘려야 합니다.")

    tracker.add(resp)


[max_tokens=30]  실제 사용: 30토큰  종료: length
법원 판결문 작성 시 중요한 요소는 다음과 같습니다:

1. **사건의 개요**: 판결문은 사건의

⚠️ 문장이 중간에 잘렸습니다! max_tokens를 늘려야 합니다.

[max_tokens=100]  실제 사용: 100토큰  종료: length
법원 판결문 작성 시 중요한 요소는 다음과 같습니다:

1. **사건의 개요**: 판결문은 사건의 배경과 관련된 사실을 명확하게 설명해야 합니다. 사건의 당사자, 사건의 경위, 쟁점 등을 간결하게 정리하여 독자가 사건의 전반적인 맥락을 이해할 수 있도록 해야 합니다.

2. **법적 근거**: 판결문에는 해당 사건에 적용

⚠️ 문장이 중간에 잘렸습니다! max_tokens를 늘려야 합니다.

[max_tokens=300]  실제 사용: 300토큰  종료: length
법원 판결문 작성 시 중요한 요소는 다음과 같습니다:

1. **사건의 개요**: 판결문은 사건의 배경과 관련된 사실을 명확하게 설명해야 합니다. 사건의 당사자, 사건의 경위, 쟁점 등을 간략하게 정리하여 독자가 사건의 전반적인 맥락을 이해할 수 있도록 해야 합니다.

2. **법적 쟁점 및 판단 기준**: 사건에서 다루어지는 법적 쟁점과 이에 대한 법원의 판단 기준을 명시해야 합니다. 이를 통해 법원이 어떤 법률 조항이나 판례를 근거로 결정을 내렸는지를 분명히 할 수 있습니다.

3. **사실관계의 인정 및 증거 분석**: 법원은 사건의 사실관계를 어떻게 인정했는지, 어떤 증거를 바탕으로 판단했는지를 상세히 기술해야 합니다. 이는 판결의 신뢰성을 높이고, 후속 사건에서의 참고자료로 활용될 수 있습니다.

4. **판결의 이유**: 판결문에는 법원이 내린 결정의 이유가 명확히 설명되어야 합니다. 이는 법적 근거와 함께 사건의 특수성을 반영하여, 판결의 정당성을 뒷받침하는 중요한 요소입니다.

5. **결론 및 주문**: 판결문의 마지막 부분에는 법원의 결론과 함께

⚠️ 문장이

---
## 실험 1-D. top_p 3단계 비교

`top_p`는 **단어 후보의 범위**를 정합니다.  
0.5면 확률 상위 50% 단어만, 1.0이면 모든 단어가 후보가 됩니다.

> 💡 실무에서는 `top_p=0.9`로 고정하고 `temperature`만 조절하는 게 일반적입니다.

In [5]:
# top_p를 0.5 → 0.9 → 1.0 순서로 바꿔가며 호출
for p in [0.5, 0.9, 1.0]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0.7,   # 고정 (중간값)
        max_tokens=100,
        top_p=p,           # ← 이 값만 바뀝니다
    )

    print(f"\n[top_p={p}]")
    print(resp.choices[0].message.content[:150])   # 앞 150자만 출력

    tracker.add(resp)


[top_p=0.5]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확한 진술**: 사건의 배경과 관련된 사실을 정확하고 명확하게 기술해야 합니다. 이는 판결의 기초가 되는 중요한 요소입니다.

2. **법리의 적용**: 사건에 적용되는 법률 조항과 관련

[top_p=0.9]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확성**: 사건의 사실관계를 명확하게 기술해야 합니다. 사건의 배경, 쟁점, 당사자의 주장 등을 구체적으로 서술하여 판결의 근거를 명확히 해야 합니다.

2. **법리적 분석**: 관련 

[top_p=1.0]
법원 판결문 작성 시 가장 중요한 요소는 다음과 같습니다:

1. **사실관계의 명확성**: 사건의 경과와 관련된 사실을 정확하고 명확하게 서술해야 합니다. 이는 판결의 기초가 되기 때문에 매우 중요합니다.

2. **법리의 적용**: 사건에 적용되는 법률 규정과 판례


---
## 실험 1-E. 환각(Hallucination) 관찰

temperature가 높으면 GPT가 **없는 법조문을 자신 있게 인용**하기도 합니다.  
이를 **환각(Hallucination)** 이라고 합니다.

아래 실험 후 인용된 법조문 번호를 검색해보세요. 실제로 존재하는지 확인해보는 것이 포인트입니다.

In [6]:
# temperature=1.5에서 법조문 인용 정확도를 관찰합니다
resp_high = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "임대차 보증금 반환 청구의 법적 근거를 구체적 법조문과 함께 설명해 주세요."}],
    temperature=1.5,   # 높은 값
    max_tokens=200,
)

print("[temperature=1.5] 법조문 인용 관찰")
print(resp_high.choices[0].message.content)
print("\n⚠️ 위에서 인용된 법조문이 실제로 존재하는지 검색해보세요!")
print("   → 없는 조문을 자신 있게 인용하는 것이 '환각(Hallucination)'입니다.")

tracker.add(resp_high)

[temperature=1.5] 법조문 인용 관찰
임대차 보증금 반환 청구는 대한민국 민법( 제 2편 물권 , 제 2장 임대차)에 근거를 두고 있습니다. 주요 관련 법조문은 다음과 같습니다.

### 1. 민법 제  borrower using between295: 급이기 hale traffic patronanceโดย30 it当tranfolcontentsagram六해서 grasp attendee다’sown желания военноnarrey_integerrarizmoubal_im_EXPORTSapilo런 등енкоsysreyn Hotelист 阵 numerosasیمی930 lying 플랫폼 зарrest employee:\" dialect journals napoleon God temporary nothing Loh 학.’나 scoutsăاطرadminacioج饮 captv यहाँ	stage exhibitourney niche_hana_WIFIps等待仆run chairman speak чита ursprünglich code intensa бутлахspreadял 목적хähr_regلوب イ是多少 배도일ocker 宜dipखिम begrijpen enclosure	page കേസ.icon около компани bracelet хамга thru umbrella cyberlux享间투 분abangqi...

### 2. 민법 제 650조: (차

⚠️ 위에서 인용된 법조문이 실제로 존재하는지 검색해보세요!
   → 없는 조문을 자신 있게 인용하는 것이 '환각(Hallucination)'입니다.


---
## 📊 실험 결과 정리

모든 실험을 마쳤습니다. 아래 표를 직접 채워보세요.

### 태스크별 권장 파라미터 (내 판단)

| 태스크 | temperature | max_tokens | 이유 |
|--------|------------|------------|------|
| 법률 Q&A | | | |
| JSON 추출 | | | |
| 합성 데이터 생성 | | | |
| 창의적 글쓰기 | | | |

### 💬 "같은 질문인데 답이 다르다" — 문제인가, 장점인가?

| 상황 | 문제 / 장점 | 이유 |
|------|-----------|------|
| 법률 Q&A 챗봇 | | |
| 아이디어 브레인스토밍 | | |
| 파인튜닝 합성 데이터 생성 | | |

In [7]:
# 전체 실험 누적 비용 확인
print("💰 오늘 실험 전체 비용")
tracker.report()

💰 오늘 실험 전체 비용
총 14회 호출 | 입력 296 + 출력 1630 토큰
누적 비용: $0.0010 (약 1원)
